# AAM Audio Playground

Use this notebook to listen to tracks from `tinyAAM` directly inside Jupyter.

It is meant for quick sanity checks while you work on preprocessing and modeling.

## Setup

This cell makes sure the repository root is importable so the notebook can use the loader code in `src.data`.

In [ ]:
from pathlib import Path
import sys

# Start from the notebook's current working directory.
repo_root = Path.cwd()
# If Jupyter launched inside notebooks/, move up to the repo root.
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# Add the repo root to the import path so `from src...` works.
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

repo_root

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

# Import the dataset wrapper used throughout the project.
from src.data import AAMDataset

## Browse tracks

Create a dataset object and inspect the tracks available in your local `tinyAAM` folder.

In [ ]:
# Point the dataset wrapper at the local tinyAAM folder.
dataset = AAMDataset(repo_root / "data/raw/tinyAAM")
# Build a summary table of the available tracks.
tracks = dataset.list_tracks()
print(f"Found {len(tracks)} tracks")
tracks.head(10)

## Listen to one mix

Change `track_id` below to try different songs.

In [ ]:
# Pick a track to listen to.
track_id = "0001"
# Load the full mix audio for that track.
audio, sr = dataset.load_audio(track_id, source="mix", sr=None, mono=True)

print("track:", track_id)
print("sample rate:", sr)
print("duration (seconds):", round(len(audio) / sr, 2))

# Display an inline audio player in the notebook.
Audio(audio, rate=sr)

## Plot the waveform

This helps you visually inspect the loudness structure of the loaded track.

In [ ]:
# Build a simple time axis for plotting the waveform.
times = np.arange(len(audio)) / sr

plt.figure(figsize=(14, 4))
plt.plot(times, audio, linewidth=0.6)
plt.title(f"Waveform for track {track_id}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## Inspect annotations while listening

This is useful for checking whether the chord labels line up with what you hear.

In [ ]:
# Load the annotation tables for the selected track.
annotations = dataset.load_annotations(track_id)

print("beatinfo preview")
display(annotations["beatinfo"].head(12))

print("segments preview")
display(annotations["segments"].head())

## Listen to stems

The AAM dataset also provides instrument stems. This section lets you audition them one by one.

In [ ]:
# Load the individual stems instead of the full mix.
stems = dataset.load_audio(track_id, source="stems", sr=None, mono=True)

# Show which stems exist for the selected track.
stem_table = pd.DataFrame(
    [
        {
            "stem_name": stem_name,
            "sample_rate": stem_sr,
            "duration_seconds": round(len(stem_audio) / stem_sr, 2),
        }
        for stem_name, stem_audio, stem_sr in stems
    ]
)
stem_table

In [ ]:
# Choose which stem to play.
stem_index = 0
stem_name, stem_audio, stem_sr = stems[stem_index]

print("playing stem:", stem_name)
Audio(stem_audio, rate=stem_sr)

## Play a short excerpt

Use this section when you want to hear just one region repeatedly instead of the full song.

In [ ]:
# Choose an excerpt in seconds.
start_time = 10.0
end_time = 18.0

# Convert the chosen time window into audio sample indices.
start_sample = int(start_time * sr)
end_sample = int(end_time * sr)
audio_excerpt = audio[start_sample:end_sample]

print(f"excerpt: {start_time:.2f}s to {end_time:.2f}s")
Audio(audio_excerpt, rate=sr)